# About
This notebook is a step by step cell run to build out the full C3S dataset from raw json sets into a .db file.

What you need are:
- src_data folder: where all the json files live
- C3SDB_schema : for sqlite schema of features
- mqn_schema : for sqlite schema of MQNs
- pred_CCS_scema : for sqlite schema of predicted CCS
- build_utils : folder for extra functions to build out the db file

In [12]:
import sqlite3
import os
import pandas as pd 
import requests

from build_utils.db_init import create_db
from build_utils.src_data import add_dataset
from build_utils.smiles import (
    load_smiles_search_cache,
    save_smiles_search_cache,
    add_smiles_to_db,
)
from build_utils.mqns import add_mqns_to_db
from build_utils.classification import label_class_byname
from build_utils.clean_src import clean_database, remove_invalid_smiles, create_clean_db


# Initialize DB
Define a new db file with schema

In [13]:
#Initialize New DB File
db_name = "C3S.db"
create_db(db_name)

#Connect to DB file
con = sqlite3.connect(db_name)
cur = con.cursor()


# Populating DB File From JSON Datasets
1. adding all the raw data
2. adding smiles structures from pubchem api
3. adding mqns
4. adding chemical classification

### adding raw json entries

In [14]:
# source datasets to include in the raw json files 
_SRC_TAGS = [
    "zhou1016",
    "zhou0817",
    "zhen0917",
    "pagl0314",
    "righ0218",
    "nich1118",
    "may_0114",
    "moll0218",
    "hine1217",
    "hine0217",
    "hine0817",
    "groe0815",
    "bijl0517",
    "stow0817",
    "hine0119",
    "leap0219",
    "blaz0818",
    # "vasi0120", exclude vasi
    "tsug0220",
    "lian0118",
    "teja0918",
    "pola0620",
    "dodd0220",
    "celm1120",
    "belo0321",
    "ross0422",
    "baker0524", #new
    "mull_1223", #new
    "palm_0424", #new
    "extra_ross0422", #new
]

In [15]:
n_entries = 0
for src_tag in _SRC_TAGS:
    n_added = add_dataset(cur, src_tag)
    n_entries += n_added
    print(f"\tsrc_tag: {src_tag} n_added: {n_added}")

print(f"\ttotal entries: {n_entries}")

	src_tag: zhou1016 n_added: 847
	src_tag: zhou0817 n_added: 451
	src_tag: zhen0917 n_added: 949
	src_tag: pagl0314 n_added: 96
	src_tag: righ0218 n_added: 106
	src_tag: nich1118 n_added: 1078
	src_tag: may_0114 n_added: 498
	src_tag: moll0218 n_added: 357
	src_tag: hine1217 n_added: 163
	src_tag: hine0217 n_added: 257
	src_tag: hine0817 n_added: 1426
	src_tag: groe0815 n_added: 131
	src_tag: bijl0517 n_added: 205
	src_tag: stow0817 n_added: 86
	src_tag: hine0119 n_added: 179
	src_tag: leap0219 n_added: 405
	src_tag: blaz0818 n_added: 429
	src_tag: tsug0220 n_added: 2950
	src_tag: lian0118 n_added: 126
	src_tag: teja0918 n_added: 173
	src_tag: pola0620 n_added: 336
	src_tag: dodd0220 n_added: 48
	src_tag: celm1120 n_added: 970
	src_tag: belo0321 n_added: 311
	src_tag: ross0422 n_added: 2510
	src_tag: baker0524 n_added: 2387
	src_tag: mull_1223 n_added: 187
	src_tag: palm_0424 n_added: 18
	src_tag: extra_ross0422 n_added: 2590
	total entries: 20269


### adding smiles structures

In [16]:
smiles_cache_file = "smiles_search_cache.json"

# if a local copy of the SMILES search cache does not exist, grab the
# built-in copy from the package
if not os.path.isfile(smiles_cache_file):
    smiles_search_cache = load_smiles_search_cache(cache_file_name=None)
else:
    # load the local copy if it exists
    smiles_search_cache = load_smiles_search_cache(
        cache_file_name=smiles_cache_file 
    )

# Initialize session
sess = requests.Session()

# Add SMILES structures to DB
n_smiles, n_requests = add_smiles_to_db(cur, sess, smiles_search_cache)
print(f"\tSMILES structures added: {n_smiles}")
print(f"\tweb requests sent: {n_requests}")
print("... done")

	(   169) CCSBASE_EDC0718C9B Reduced nicotinamide adenine dinucleotide (NADH)                                                    An error occurred: 404 Client Error: PUGREST.NotFound for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Reduced%20nicotinamide%20adenine%20dinucleotide%20(NADH)/cids/TXT
An error occurred: 404 Client Error: PUGREST.NotFound for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Reduced%20nicotinamide%20adenine%20dinucleotide%20(NADH)/cids/TXT
	(   460) CCSBASE_13E7051CFC D-(+)-Melibiose                                                                                     An error occurred: 404 Client Error: PUGREST.NotFound for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/D-(+)-Melibiose/cids/TXT
An error occurred: 404 Client Error: PUGREST.NotFound for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/D-(+)-Melibiose/cids/TXT
	(   655) CCSBASE_2B2853E510 P1,P4-Diadenosine 5'-tetraphosphate                    

### adding mqn

In [17]:
print("adding MQNs to database entries ...")
n_mqns = add_mqns_to_db(cur)
print(f"\tentries with MQNs: {n_mqns}")
print("... done")

adding MQNs to database entries ...
	entries with MQNs: 18215
... done


### adding chemical labels 

In [18]:
print("adding rough chemical classification labels to database entries ...")
label_class_byname(cur)

adding rough chemical classification labels to database entries ...


In [19]:
con.commit()
con.close()

# Making Cleaned DB version
1. cleaning invalid smiles structures
2. clean code by DT and TW values
3. cleaning by relative standard deviation


In [20]:
#open c3s db file 
db_name = "C3S.db"
clean_db_name = "C3S_clean_smile.db"

create_clean_db(clean_db_name)

In [21]:
# 1. remove invalid smiles
remove_invalid_smiles(db_name, clean_db_name) #updates the clean DB with only valid smiles

In [22]:
# 2. clean code by DT and TW values and rsd
clean_database("C3S_clean_smile.db", "C3S_clean_smile_rsd.db")

Creating new clean database: C3S_clean_smile_rsd.db
🔴 Entries processed: 0/14496 🔴 
🔴 Entries processed: 1/14496 🔴 
🔴 Entries processed: 2/14496 🔴 
🔴 Entries processed: 3/14496 🔴 
🔴 Entries processed: 4/14496 🔴 
🔴 Entries processed: 5/14496 🔴 
🔴 Entries processed: 6/14496 🔴 
🔴 Entries processed: 7/14496 🔴 
✅ Processing group with key: ('(hex)10', '[M+H]+', np.float64(1640.0))                     g_id     name  adduct     mass  z       mz  ccs  \
3708  CCSBASE_7A8C7F3B80  (hex)10  [M+H]+  1639.55  1  1639.55  365   
3709  CCSBASE_C789C0B6F4  (hex)10  [M+H]+  1639.55  1  1639.55  390   

                                                    smi chem_class_label  \
3708  C(C1C(C(C(C(O1)OC2C(OC(C(C2O)O)OC3C(OC(C(C3O)O...     carbohydrate   
3709  C(C1C(C(C(C(O1)OC2C(OC(C(C2O)O)OC3C(OC(C(C3O)O...     carbohydrate   

       src_tag  ... r4 r5  r6  r7  r8  r9  rg10  afr  bfr  rounded_mz  
3708  may_0114  ...  0  0  10   0   0   0     0    0    0      1640.0  
3709  may_0114  ...  0  0  10   0 